# 技能3 · Day 5 上机：规模实验与营销应用（MAB + CATE 综合案例）

**版本**：v5.0 学习材料包
**配套**：notes.md（讲义）｜ data/README.md（真实数据集）｜ solution.ipynb（参考答案，做完再看）

## 学习目标
学完你能：
1. 用 Thompson Sampling MAB 在真实响应率数据上对比固定 A/B，理解自适应实验的探索-利用权衡
2. 用 econml CausalForestDML 估计异质处理效应（CATE），识别响应最大的用户群
3. 完成"数据->因果->决策"综合案例流程
4. 识别幸存者偏差、辛普森悖论、选择性停止三大因果陷阱

## 说明
本笔记本有 **6 个 TODO**。真实数据集：NSW（综合案例 + CATE + MAB 真实响应率驱动）。

## 0. 环境准备
首次运行需安装依赖（取消注释执行一次）：

In [ ]:
# !pip install causaldata dowhy econml scikit-learn scipy -q

## 1. 综合案例背景与营销映射

**场景**：你是营销产品经理，评估"AI个性化推荐/优惠券"是否值得全量上线，且对谁全量。

**NSW 真实数据驱动综合案例**：
- 处理 T = `treat`（是否收到优惠券/AI推荐）
- 结果 Y = `re78`（GMV/转化）
- 协变量 X = `age/education/re74/re75/...`（用户画像，CATE 用）
- MAB 响应率 = `re78>0` 比例（真实转化率驱动 bandit）

**流程**：固定 A/B 基线(ATE) -> MAB 自适应(省实验成本) -> CATE(对谁全量) -> 反驳(稳健性) -> 陷阱检测

In [ ]:
import pandas as pd
import numpy as np
from scipy import stats
from scipy.stats import beta as beta_dist
import dowhy
from dowhy import CausalModel
from causaldata import nsw
from econml.dml import CausalForestDML
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
import warnings
warnings.filterwarnings('ignore')

## TODO 1-2：加载真实数据 + 固定 A/B 基线

In [ ]:
# TODO 1：加载真实 NSW 数据，准备综合案例
# 提示：from causaldata import nsw; df = nsw.load_pandas().data
# 要求：加载 df，定义 T/Y/X，打印形状 + 处理/对照样本量

# ===== 你的代码 =====
df = None  # TODO
# ====================

T = df['treat']; Y = df['re78']
X = df[['age','education','black','hispanic','married','nodegree','re74','re75']]
print(f"形状: {df.shape}, 处理组: {(df['treat']==1).sum()}, 对照组: {(df['treat']==0).sum()}")

In [ ]:
# TODO 2：固定 A/B 基线分析（均值差 + t 检验）
# 提示：ate = 处理组 re78 均值 - 对照组 re78 均值
#       stats.ttest_ind(处理组 re78, 对照组 re78)
# 要求：打印 ATE、t 值、p 值、是否显著

# ===== 你的代码 =====
ate = None  # TODO
t_stat = None  # TODO
p_val = None  # TODO
# ====================

print(f"固定 A/B 估计 ATE = {ate:.2f}")
print(f"t={t_stat:.4f}, p={p_val:.4f}, 显著: {'是' if p_val<0.05 else '否'}")

## 2. 多臂老虎机（MAB）与自适应实验

传统 A/B 固定分配，实验期间部分用户分到较差版本，造成实验成本。**Thompson Sampling**：对每臂维护 Beta 后验，采样选最大，观察后更新，自动平衡探索-利用。

$$\text{Thompson: 采样 } \theta_k \sim \text{Beta}(\alpha_k, \beta_k), \text{选 } \arg\max_k \theta_k$$

**本 Day 用真实响应率驱动**：把 NSW 按 age 分5组，各组 `re78>0` 的真实比例作为5个"广告创意"臂的真实 CTR。对比固定 A/B（每臂等量）vs Thompson Sampling 的累计转化--MAB 把更多流量分给高响应组，节省实验成本。

## TODO 3：Thompson Sampling MAB

In [ ]:
# TODO 3：Thompson Sampling MAB -- 用 NSW 真实响应率驱动，对比固定 A/B
# 提示：
#   df['responded'] = (df['re78']>0).astype(int)
#   df['age_group'] = pd.cut(df['age'], bins=5, labels=False)
#   true_rates = df.groupby('age_group')['responded'].mean().values  # 5臂真实响应率
#   Thompson: alpha=beta=ones(5), 每步采样 Beta(alpha[i],beta[i]) 选argmax, 观察 reward 更新
# 要求：打印真实响应率、固定A/B总转化、Thompson总转化、MAB优势、各臂被选次数

# ===== 你的代码 =====
true_rates = None  # TODO
ts_rewards = None  # TODO
ts_pulls = None  # TODO
# ====================

np.random.seed(42)
ab_rewards = sum(np.random.binomial(1, true_rates[a], 1000).sum() for a in range(5))
print(f"真实响应率: {true_rates}")
print(f"固定 A/B 总转化: {ab_rewards}")
print(f"Thompson Sampling 总转化: {ts_rewards}")
print(f"MAB 优势: +{ts_rewards-ab_rewards} ({(ts_rewards/max(ab_rewards,1)-1)*100:.1f}%)")
print(f"各臂被选次数: {ts_pulls.astype(int)}")

## 3. 异质处理效应（CATE）

ATE 是平均效应，但不同用户响应不同。**CausalForestDML**（econml）估计 CATE：哪类用户对处理响应最大。

**营销决策**：ATE 判断"要不要全量"，CATE 判断"对谁全量"--找响应最大的用户群精准投放，而非全量发券。

## TODO 4-5：CATE 异质效应 + 反驳检验

In [ ]:
# TODO 4：CATE -- econml CausalForestDML 估计异质处理效应
# 提示：
#   est = CausalForestDML(model_y=RandomForestRegressor(n_estimators=50,min_samples_leaf=5,random_state=42),
#                         model_t=RandomForestClassifier(n_estimators=50,min_samples_leaf=5,random_state=42),
#                         n_estimators=100, min_samples_leaf=5, random_state=42)
#   est.fit(Y_arr, T_arr, X=X_arr); cate = est.effect(X_arr)
# 要求：打印平均 CATE、CATE 标准差、各年龄组 CATE

# ===== 你的代码 =====
cate = None  # TODO
# ====================

df['cate'] = cate
print(f"平均 CATE = {cate.mean():.2f} (对比 ATE {ate:.2f})")
print(f"CATE 标准差 = {cate.std():.2f} (异质程度)")
print("\n各年龄组 CATE:")
print(df.groupby('age_group')['cate'].mean())

In [ ]:
# TODO 5：反驳检验（安慰剂处理）-- 验证估计稳健性
# 提示：
#   model = CausalModel(data=df, treatment='treat', outcome='re78',
#                       common_causes=['age','education','black','hispanic','married','nodegree','re74','re75'])
#   identified = model.identify_effect()
#   estimate = model.estimate_effect(identified, method_name='backdoor.linear_regression')
#   refutation = model.refute_estimate(identified, estimate, 'placebo_treatment_refuter')
# 要求：打印安慰剂反驳结果

# ===== 你的代码 =====
refutation = None  # TODO
# ====================

print(refutation)

## TODO 6（可选）：辛普森悖论检测

In [ ]:
# TODO 6（可选）：辛普森悖论检测 -- 按 black 分群看 ATE 是否反转
# 提示：对 black=0 和 black=1 子群分别算 treat 组-对照组 re78 均值差
# 要求：打印各子群 ATE + 总体 ATE，判断是否反转

# ===== 你的代码 =====

# ====================

print(f"总体 ATE = {ate:.2f}")
print("若分群效应方向与总体相反，提示辛普森悖论")

## 5. 反思与前沿

### 反思问题
1. MAB vs 固定 A/B 在你的真实响应率数据上节省了多少转化？这与"实验成本"概念如何对应？
2. CATE 的标准差多大？说明用户响应的异质程度如何？哪个年龄组响应最大？
3. 安慰剂检验是否支持 ATE 估计的稳健性？
4. 分群后 ATE 方向是否与总体一致？若反转，是否辛普森悖论？根源是什么？

### 2026 前沿：Uplift Modeling（增量建模 + Qini）
CATE 找"响应最大"的用户，**Uplift Modeling** 进一步把用户分四类--可被说服/必然转化/必不转化/反响应--只投"可被说服"群体，相同预算下增量转化最大化。用 `scikit-uplift` 画 Qini 曲线评估。
参考 arXiv 1603.05824（Gutierrez & Gérardy 2017）。

> 🔗 深入阅读见 `reading.md` 的 Uplift 条目。